In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
model_id = "HuggingFaceTB/SmolLM2-1.7B"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto"
)

print("Model and tokenizer loaded.")

Some parameters are on the meta device because they were offloaded to the cpu.


Model and tokenizer loaded.


# LogitLens VS Tuned Lens on a simple example
We use a small model (SmolLM2) and a simple prompt to illustrate the difference between an LogitLens (untrained Tuned Lens) and a Tuned Lens (trained Tuned Lens). As we will see, LogitLens is not able to predict the next token correctly until the penultimate (22th) layer, while Tuned Lens is able to predict the next token correctly from the 18th layer. It shows two things: 
1. The model already has the information about the next token in its hidden states, but it is not accessible to the LogitLens until the penultimate layer. 
2. The Tuned Lens is able to extract this information from the hidden states and make a correct prediction earlier in the model.

Note that logit lens is a special case of tuned lens, where the weights matrix are identity matrices and bias vectors are zero vectors (default values of SmoleLMTunedLens).

In [9]:
import pandas as pd
from IPython.display import display
from tuned_lens_model import SmolLMTunedLens

# Instantiate untrained Tuned Lens, i.e. a LogitLens
device = model.device
untrained_lens = SmolLMTunedLens(model)
untrained_lens.translators.to(device)
untrained_lens.eval()

# Prompt
prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

print("Prediction for the prompt using LogitLens:")
print(f"Prompt: '{prompt}'")
print("-" * 50)

# Run the base model to extract hidden states
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

all_hidden_states = outputs.hidden_states
results = []

# Predict the next token layer by layer
for layer_idx in range(untrained_lens.num_layers - 1):
    # Get the output state of the current layer
    h_state = all_hidden_states[layer_idx + 1]
    
    # Pass it through the untrained translator
    with torch.no_grad():
        logits = untrained_lens(h_state, layer_idx)
    
    # We only care about the very last token's predictions (to guess the next word)
    next_token_logits = logits[0, -1, :]
    
    # Get the highest probability token
    probs = torch.softmax(next_token_logits, dim=-1)
    top_token_id = torch.argmax(probs).item()
    top_token_str = tokenizer.decode(top_token_id)
    top_prob = probs[top_token_id].item()
    
    # Save the result
    results.append({
        "Layer": f"Layer {layer_idx}",
        "Predicted Next Token": repr(top_token_str), # repr() to see hidden spaces/newlines
        "Confidence": f"{top_prob:.2%}"
    })

df_predictions = pd.DataFrame(results)
display(df_predictions)

Prediction for the prompt using LogitLens:
Prompt: 'The capital of France is'
--------------------------------------------------


,Layer,Predicted Next Token,Confidence
0,Layer 0,' subdu',22.17%
1,Layer 1,' neighb',60.97%
2,Layer 2,' neighb',54.17%
3,Layer 3,' neighb',78.89%
4,Layer 4,' neighb',47.74%
5,Layer 5,' pries',25.44%
6,Layer 6,' pries',63.19%
7,Layer 7,' pries',93.82%
8,Layer 8,' pries',33.89%
9,Layer 9,' immers',38.58%


In [10]:
import os
from pathlib import Path
# Instantiate untrained Tuned Lens, i.e. a LogitLens
device = model.device
trained_lens = SmolLMTunedLens(model)
chkpt_path = Path("..") / "checkpoints" / "smollm2_tuned_lens_5000.pt"
state_dict = torch.load(chkpt_path, weights_only=True)
trained_lens.load_state_dict(state_dict, strict=False)
trained_lens.translators.to(device)
trained_lens.eval()

# Prompt
prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

print("Prediction for the prompt using Tuned Lens:")
print(f"Prompt: '{prompt}'")
print("-" * 50)

# Run the base model to extract hidden states
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

all_hidden_states = outputs.hidden_states
results = []

# Predict the next token layer by layer
for layer_idx in range(trained_lens.num_layers - 1):
    # Get the output state of the current layer
    h_state = all_hidden_states[layer_idx + 1]
    
    # Pass it through the untrained translator
    with torch.no_grad():
        logits = trained_lens(h_state, layer_idx)
    
    # We only care about the very last token's predictions (to guess the next word)
    next_token_logits = logits[0, -1, :]
    
    # Get the highest probability token
    probs = torch.softmax(next_token_logits, dim=-1)
    top_token_id = torch.argmax(probs).item()
    top_token_str = tokenizer.decode(top_token_id)
    top_prob = probs[top_token_id].item()
    
    # Save the result
    results.append({
        "Layer": f"Layer {layer_idx}",
        "Predicted Next Token": repr(top_token_str), # repr() to see hidden spaces/newlines
        "Confidence": f"{top_prob:.2%}"
    })

df_predictions = pd.DataFrame(results)
display(df_predictions)

c:\Users\baudo\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\module.py:2588: UserWarning: for final_norm.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(


Prediction for the prompt using Tuned Lens:
Prompt: 'The capital of France is'
--------------------------------------------------


,Layer,Predicted Next Token,Confidence
0,Layer 0,' the',13.01%
1,Layer 1,' the',22.55%
2,Layer 2,' the',16.72%
3,Layer 3,' a',17.13%
4,Layer 4,' the',13.73%
5,Layer 5,' the',12.98%
6,Layer 6,' the',18.57%
7,Layer 7,' the',12.23%
8,Layer 8,' the',8.37%
9,Layer 9,' the',8.10%
